In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# ==========================================
# 1. Parse and Plot the Complex Energy File
# ==========================================
def plot_energy_over_time(energy_file):
    print(f"Processing {energy_file}...")
    
    # The energy file has 2 lines of preamble. We skip them to get to the actual header.
    df_energy = pd.read_csv(energy_file, skiprows=2)
    
    # Create the plot
    plt.figure(figsize=(12, 6))
    
    # Plot Total Energy as a prominent line
    plt.plot(df_energy['Frame #'], df_energy['TOTAL'], 
             label='Total Energy', color='black', linewidth=2)
    
    # Plot components as slightly transparent lines
    plt.plot(df_energy['Frame #'], df_energy['VDWAALS'], label='van der Waals', alpha=0.7)
    plt.plot(df_energy['Frame #'], df_energy['EEL'], label='Electrostatic', alpha=0.7)
    plt.plot(df_energy['Frame #'], df_energy['EGB'], label='Polar Solvation (EGB)', alpha=0.7)
    
    plt.xlabel('Frame #', fontsize=12)
    plt.ylabel('Energy (kcal/mol)', fontsize=12)
    plt.title('MM-GBSA Energy Components over Time', fontsize=14, fontweight='bold')
    plt.legend(loc='best')
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    
    # Save and show
    plt.savefig('energy_over_time.png', dpi=300)
    print("Saved 'energy_over_time.png'")
    plt.show()

# ==========================================
# 2. Parse and Plot the Decomposition File
# ==========================================
def plot_residue_decomposition(decomp_file):
    print(f"Processing {decomp_file}...")
    
    # The decomposition file has a messy multi-row header. 
    # It's safest to skip the first 8 rows entirely and define the columns manually.
    decomp_cols = [
        'Residue', 'Location',
        'Internal_Avg', 'Internal_StdDev', 'Internal_StdErr',
        'vdW_Avg', 'vdW_StdDev', 'vdW_StdErr',
        'Elec_Avg', 'Elec_StdDev', 'Elec_StdErr',
        'PolSolv_Avg', 'PolSolv_StdDev', 'PolSolv_StdErr',
        'NonPolSolv_Avg', 'NonPolSolv_StdDev', 'NonPolSolv_StdErr',
        'Total_Avg', 'Total_StdDev', 'Total_StdErr'
    ]
    
    df_decomp = pd.read_csv(decomp_file, skiprows=8, names=decomp_cols)
    
    # Clean up the Residue names (e.g., changing 'PRO   1' to 'PRO 1' for a cleaner x-axis)
    df_decomp['Residue'] = df_decomp['Residue'].str.replace(r'\s+', ' ', regex=True)
    
    # Filter out any lingering malformed text rows if they exist at the bottom of the file
    df_decomp = df_decomp[pd.to_numeric(df_decomp['Total_Avg'], errors='coerce').notnull()]
    df_decomp['Total_Avg'] = df_decomp['Total_Avg'].astype(float)
    df_decomp['Total_StdDev'] = df_decomp['Total_StdDev'].astype(float)
    
    # Create the plot
    plt.figure(figsize=(16, 6))
    
    # Use seaborn for a nice barplot
    sns.barplot(data=df_decomp, x='Residue', y='Total_Avg', color='royalblue', 
                edgecolor='black', zorder=2)
    
    # Add error bars manually using matplotlib (seaborn's built-in requires raw data, not pre-calculated stddev)
    plt.errorbar(x=np.arange(len(df_decomp)), y=df_decomp['Total_Avg'], 
                 yerr=df_decomp['Total_StdDev'], fmt='none', c='black', capsize=3, zorder=3)
    
    # Formatting
    plt.axhline(0, color='black', linewidth=1, zorder=1) # Line at y=0
    plt.xticks(rotation=90, fontsize=8) # Rotate labels so they don't overlap
    plt.xlabel('Residue', fontsize=12)
    plt.ylabel('Total Energy Contribution (kcal/mol)', fontsize=12)
    plt.title('Per-Residue Energy Decomposition', fontsize=14, fontweight='bold')
    plt.grid(axis='y', linestyle='--', alpha=0.7, zorder=0)
    plt.tight_layout()
    
    # Save and show
    plt.savefig('per_residue_decomposition.png', dpi=300)
    print("Saved 'per_residue_decomposition.png'")
    plt.show()

# ==========================================
# 3. Execute the functions
# ==========================================
if __name__ == "__main__":
    energy_file = 'energy_2BPW_fbe.csv'
    decomp_file = 'decomp_2BPW_fbe.csv'
    
    try:
        plot_energy_over_time(energy_file)
    except FileNotFoundError:
        print(f"Could not find {energy_file}. Check the file name and path.")
        
    try:
        plot_residue_decomposition(decomp_file)
    except FileNotFoundError:
        print(f"Could not find {decomp_file}. Check the file name and path.")